In [ ]:
# Gridflow silver data inspection notebook
#
# Purpose:
# - inspect silver parquet outputs
# - validate coverage, missingness, and row counts
# - visualise load / generation / price for a sample of zones
# - help confirm the bronze->silver pipeline is doing what we think it is doing

In [ ]:
from pathlib import Path
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

In [ ]:
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
SILVER_ROOT = DATA_ROOT / "silver"

ENTSOE_SILVER_ROOT = SILVER_ROOT / "entsoe"
ELEXON_SILVER_ROOT = SILVER_ROOT / "elexon"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SILVER_ROOT:", SILVER_ROOT)
print("ENTSOE_SILVER_ROOT exists:", ENTSOE_SILVER_ROOT.exists())
print("ELEXON_SILVER_ROOT exists:", ELEXON_SILVER_ROOT.exists())

In [ ]:
def parquet_files_under(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(root.rglob("*.parquet"))


def extract_date_from_filename(path: Path) -> str | None:
    m = re.search(r"(\d{4}-\d{2}-\d{2})", path.name)
    return m.group(1) if m else None


def safe_read_parquet(path: Path) -> pd.DataFrame:
    try:
        return pd.read_parquet(path)
    except Exception as exc:
        print(f"FAILED reading {path}: {exc}")
        return pd.DataFrame()

In [ ]:
entsoe_files = parquet_files_under(ENTSOE_SILVER_ROOT)
elexon_files = parquet_files_under(ELEXON_SILVER_ROOT)

print("ENTSOE parquet files:", len(entsoe_files))
print("ELEXON parquet files:", len(elexon_files))

print("\nSample ENTSOE files:")
for p in entsoe_files[:10]:
    print(" ", p)

print("\nSample ELEXON files:")
for p in elexon_files[:10]:
    print(" ", p)

In [ ]:
def build_entsoe_inventory(root: Path) -> pd.DataFrame:
    rows = []

    for path in parquet_files_under(root):
        rel = path.relative_to(root)
        parts = rel.parts

        # expected: <zone>/<dataset>/year=YYYY/month=MM/file.parquet
        if len(parts) < 5:
            continue

        zone = parts[0]
        dataset = parts[1]
        year_part = parts[2]
        month_part = parts[3]
        day_str = extract_date_from_filename(path)

        rows.append(
            {
                "zone": zone,
                "dataset": dataset,
                "year_part": year_part,
                "month_part": month_part,
                "date": day_str,
                "path": str(path),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values(["zone", "dataset", "date"]).reset_index(drop=True)


entsoe_inventory = build_entsoe_inventory(ENTSOE_SILVER_ROOT)
entsoe_inventory.head(20)

In [ ]:
def build_elexon_inventory(root: Path) -> pd.DataFrame:
    rows = []

    for path in parquet_files_under(root):
        rel = path.relative_to(root)
        parts = rel.parts

        # expected: <dataset>/year=YYYY/month=MM/file.parquet
        if len(parts) < 4:
            continue

        dataset = parts[0]
        year_part = parts[1]
        month_part = parts[2]
        day_str = extract_date_from_filename(path)

        rows.append(
            {
                "dataset": dataset,
                "year_part": year_part,
                "month_part": month_part,
                "date": day_str,
                "path": str(path),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values(["dataset", "date"]).reset_index(drop=True)


elexon_inventory = build_elexon_inventory(ELEXON_SILVER_ROOT)
elexon_inventory.head(20)

## coverage summary tables

In [ ]:
if not entsoe_inventory.empty:
    entsoe_coverage = (
        entsoe_inventory.groupby(["zone", "dataset"])
        .agg(
            files=("path", "count"),
            min_date=("date", "min"),
            max_date=("date", "max"),
        )
        .reset_index()
        .sort_values(["zone", "dataset"])
    )
    display(entsoe_coverage)
else:
    print("No ENTSOE inventory found.")

if not elexon_inventory.empty:
    elexon_coverage = (
        elexon_inventory.groupby(["dataset"])
        .agg(
            files=("path", "count"),
            min_date=("date", "min"),
            max_date=("date", "max"),
        )
        .reset_index()
        .sort_values(["dataset"])
    )
    display(elexon_coverage)
else:
    print("No ELEXON inventory found.")

## Load all entso-e/elexon data for a single zone

In [ ]:
def load_entsoe_zone_dataset(zone: str, dataset: str) -> pd.DataFrame:
    paths = entsoe_inventory.loc[
        (entsoe_inventory["zone"] == zone) & (entsoe_inventory["dataset"] == dataset),
        "path",
    ].tolist()

    dfs = []
    for p in paths:
        df = safe_read_parquet(Path(p))
        if not df.empty:
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)

    if "timestamp_utc" in out.columns:
        out["timestamp_utc"] = pd.to_datetime(out["timestamp_utc"], utc=True, errors="coerce")

    return out


def load_elexon_dataset(dataset: str) -> pd.DataFrame:
    paths = elexon_inventory.loc[elexon_inventory["dataset"] == dataset, "path"].tolist()

    dfs = []
    for p in paths:
        df = safe_read_parquet(Path(p))
        if not df.empty:
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)

    if "timestamp_utc" in out.columns:
        out["timestamp_utc"] = pd.to_datetime(out["timestamp_utc"], utc=True, errors="coerce")

    if "publish_time_utc" in out.columns:
        out["publish_time_utc"] = pd.to_datetime(out["publish_time_utc"], utc=True, errors="coerce")

    return out

## FR & GB data loads

In [ ]:
fr_load = load_entsoe_zone_dataset("fr", "actual_total_load")
fr_gen = load_entsoe_zone_dataset("fr", "generation_per_type")
fr_price = load_entsoe_zone_dataset("fr", "energy_prices")

gb_load = load_elexon_dataset("demand_actual_total")
gb_gen = load_elexon_dataset("fuelhh")
gb_price = load_elexon_dataset("mid")

print("FR load:", fr_load.shape)
print("FR generation:", fr_gen.shape)
print("FR prices:", fr_price.shape)

print("GB load:", gb_load.shape)
print("GB generation:", gb_gen.shape)
print("GB prices:", gb_price.shape)

In [ ]:
def inspect_df(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f"\n{name}")
    print("-" * len(name))
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    if not df.empty:
        display(df.head(n))


inspect_df("FR load", fr_load)
inspect_df("FR generation", fr_gen)
inspect_df("FR prices", fr_price)

inspect_df("GB load", gb_load)
inspect_df("GB generation", gb_gen)
inspect_df("GB prices", gb_price)

## Missing data summary

In [ ]:
def missingness_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["column", "null_count", "null_pct"])

    out = pd.DataFrame(
        {
            "column": df.columns,
            "null_count": [df[c].isna().sum() for c in df.columns],
        }
    )
    out["null_pct"] = out["null_count"] / len(df)
    return out.sort_values(["null_pct", "null_count"], ascending=False).reset_index(drop=True)


display(missingness_summary(fr_load))
display(missingness_summary(fr_gen))
display(missingness_summary(fr_price))

In [ ]:
def daily_row_counts_entsoe(zone: str, dataset: str) -> pd.DataFrame:
    df = load_entsoe_zone_dataset(zone, dataset)
    if df.empty or "timestamp_utc" not in df.columns:
        return pd.DataFrame(columns=["date", "rows"])

    out = (
        df.assign(date=df["timestamp_utc"].dt.date)
        .groupby("date")
        .size()
        .rename("rows")
        .reset_index()
    )
    return out


zones_to_check = sorted(entsoe_inventory["zone"].unique()) if not entsoe_inventory.empty else []

for zone in zones_to_check[:8]:
    print(f"\nZONE: {zone}")
    for dataset in ["actual_total_load", "generation_per_type", "energy_prices"]:
        counts = daily_row_counts_entsoe(zone, dataset)
        if counts.empty:
            print(f"  {dataset}: no data")
        else:
            print(
                f"  {dataset}: min_rows={counts['rows'].min()} "
                f"max_rows={counts['rows'].max()} "
                f"days={len(counts)}"
            )

In [ ]:
def coverage_table_entsoe(inventory: pd.DataFrame) -> pd.DataFrame:
    if inventory.empty:
        return pd.DataFrame()

    out = (
        inventory.assign(flag=1)
        .pivot_table(
            index=["zone", "dataset"],
            columns=inventory["date"].dt.strftime("%Y-%m-%d"),
            values="flag",
            aggfunc="max",
            fill_value=0,
        )
        .sort_index()
    )
    return out


coverage_matrix = coverage_table_entsoe(entsoe_inventory)
coverage_matrix.iloc[:20, :20]

In [ ]:
def plot_time_series(df: pd.DataFrame, x: str, y: str, title: str, start=None, end=None):
    plot_df = df.copy()

    if plot_df.empty:
        print(f"{title}: no data")
        return

    if start is not None:
        plot_df = plot_df[plot_df[x] >= pd.Timestamp(start, tz="UTC")]
    if end is not None:
        plot_df = plot_df[plot_df[x] < pd.Timestamp(end, tz="UTC")]

    if plot_df.empty:
        print(f"{title}: no data in selected window")
        return

    plt.figure(figsize=(12, 4))
    plt.plot(plot_df[x], plot_df[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.show()

## France plots

In [ ]:
plot_time_series(
    fr_load.sort_values("timestamp_utc"),
    x="timestamp_utc",
    y="load_mw",
    title="France actual total load",
)

In [ ]:
plot_time_series(
    fr_price.sort_values("timestamp_utc"),
    x="timestamp_utc",
    y="price",
    title="France energy prices",
)

In [ ]:
if not fr_gen.empty:
    gen_pivot = (
        fr_gen.groupby(["timestamp_utc", "psr_type"], as_index=False)["generation_mw"]
        .sum()
        .pivot(index="timestamp_utc", columns="psr_type", values="generation_mw")
        .sort_index()
    )

    top_psr = gen_pivot.sum().sort_values(ascending=False).head(8).index.tolist()

    plt.figure(figsize=(12, 5))
    for psr in top_psr:
        plt.plot(gen_pivot.index, gen_pivot[psr], label=psr)
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.title("France generation by top production types")
    plt.xlabel("timestamp_utc")
    plt.ylabel("generation_mw")
    plt.tight_layout()
    plt.show()
else:
    print("No France generation data.")

## GB plots

In [ ]:
plot_time_series(
    gb_load.sort_values("timestamp_utc"),
    x="timestamp_utc",
    y="demand_mw",
    title="GB demand_actual_total",
)

In [ ]:
# Keep provider rows separate first
gb_price_sorted = gb_price.sort_values("timestamp_utc")

for provider in sorted(gb_price_sorted["data_provider"].dropna().unique()):
    subset = gb_price_sorted[gb_price_sorted["data_provider"] == provider]
    plt.figure(figsize=(12, 4))
    plt.plot(subset["timestamp_utc"], subset["price_gbp_mwh"])
    plt.title(f"GB MID price - {provider}")
    plt.xlabel("timestamp_utc")
    plt.ylabel("price_gbp_mwh")
    plt.tight_layout()
    plt.show()

In [ ]:
if not gb_gen.empty:
    gb_gen_pivot = (
        gb_gen.groupby(["timestamp_utc", "fuel_type"], as_index=False)["generation_mw"]
        .sum()
        .pivot(index="timestamp_utc", columns="fuel_type", values="generation_mw")
        .sort_index()
    )

    top_fuels = gb_gen_pivot.sum().sort_values(ascending=False).head(8).index.tolist()

    plt.figure(figsize=(12, 5))
    for fuel in top_fuels:
        plt.plot(gb_gen_pivot.index, gb_gen_pivot[fuel], label=fuel)
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.title("GB generation by top fuel types")
    plt.xlabel("timestamp_utc")
    plt.ylabel("generation_mw")
    plt.tight_layout()
    plt.show()
else:
    print("No GB generation data.")

## Quick comparison: GB vs FR

In [ ]:
fr_load_daily = (
    fr_load.groupby(fr_load["timestamp_utc"].dt.date)["load_mw"]
    .mean()
    .rename("fr_load_mean")
)

gb_load_daily = (
    gb_load.groupby(gb_load["timestamp_utc"].dt.date)["demand_mw"]
    .mean()
    .rename("gb_load_mean")
)

load_compare = pd.concat([fr_load_daily, gb_load_daily], axis=1).reset_index().rename(columns={"timestamp_utc": "date"})
display(load_compare.head())

plt.figure(figsize=(12, 4))
plt.plot(load_compare["date"], load_compare["fr_load_mean"], label="FR mean load")
plt.plot(load_compare["date"], load_compare["gb_load_mean"], label="GB mean load")
plt.legend()
plt.title("France vs GB daily mean load")
plt.xlabel("date")
plt.ylabel("MW")
plt.tight_layout()
plt.show()

## Daily expected-row validation for ENTSO-E load

In [ ]:
def expected_rows_from_resolution(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["date", "resolution", "rows"])

    out = (
        df.assign(
            date=df["timestamp_utc"].dt.date,
            resolution=df["resolution"].astype(str),
        )
        .groupby(["date", "resolution"])
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["date", "resolution"])
    )
    return out


fr_load_expected = expected_rows_from_resolution(fr_load)
display(fr_load_expected.head(20))

## Zone level summary friction

In [ ]:
def summarise_entsoe_zone(zone: str) -> pd.DataFrame:
    rows = []

    for dataset in ["actual_total_load", "generation_per_type", "energy_prices"]:
        df = load_entsoe_zone_dataset(zone, dataset)

        rows.append(
            {
                "zone": zone,
                "dataset": dataset,
                "rows": len(df),
                "min_ts": df["timestamp_utc"].min() if "timestamp_utc" in df.columns and not df.empty else pd.NaT,
                "max_ts": df["timestamp_utc"].max() if "timestamp_utc" in df.columns and not df.empty else pd.NaT,
                "columns": ", ".join(df.columns[:8]) + (" ..." if len(df.columns) > 8 else ""),
            }
        )

    return pd.DataFrame(rows)


zone_summaries = pd.concat([summarise_entsoe_zone(z) for z in sorted(entsoe_inventory["zone"].unique())], ignore_index=True)
display(zone_summaries)

## Find suspicious days

In [ ]:
def suspicious_days(zone: str, dataset: str, min_rows_threshold: int = 1) -> pd.DataFrame:
    counts = daily_row_counts_entsoe(zone, dataset)
    if counts.empty:
        return counts

    return counts[counts["rows"] <= min_rows_threshold].sort_values("date").reset_index(drop=True)


display(suspicious_days("fr", "actual_total_load", min_rows_threshold=10))
display(suspicious_days("ie_sem", "actual_total_load", min_rows_threshold=10))

## Optional: Explore problematic data frame

In [ ]:
example_path = None

subset = entsoe_inventory[
    (entsoe_inventory["zone"] == "fr") &
    (entsoe_inventory["dataset"] == "actual_total_load")
]

if not subset.empty:
    example_path = Path(subset.iloc[0]["path"])
    print(example_path)

if example_path:
    df_example = pd.read_parquet(example_path)
    display(df_example.head(20))
    print(df_example.columns.tolist())
    print("rows:", len(df_example))